# Kámárí · 03 · EDA  (Google Colab)

Understand the manifest before training, focused on **safety + fairness**:

- age distribution and the **13–21 boundary**
- **skin-band** coverage (dark-skin representation)
- per-dataset / gender balance

Writes figures + `data_quality_report.md` locally. Upload happens in notebook 04.

In [ ]:
# --- Kámárí bootstrap: works on Google Colab and locally ---
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not pathlib.Path('/content/kamari/data').exists():
        try:
            from google.colab import userdata
            os.environ.setdefault('KAMARI_REPO_URL', userdata.get('KAMARI_REPO_URL'))
        except Exception: pass
        os.system(f"git clone {os.environ.get('KAMARI_REPO_URL','')} /content/kamari")
    os.system('pip install -q -r /content/kamari/requirements-data.txt')
    REPO = pathlib.Path('/content/kamari')
else:
    REPO = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (c/'data'/'registry').exists()), pathlib.Path.cwd().parent)
sys.path.append(str(REPO))
print('REPO =', REPO, '| Colab:', IN_COLAB)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
MAN = REPO / 'data' / 'manifests'
FIG = REPO / 'data' / 'cards' / 'eda'; FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(MAN / 'manifest_public_v0.parquet')
print('rows:', len(df)); df.head()

In [ ]:
# Age distribution + 13–21 boundary band
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['age'].dropna().hist(bins=40, ax=ax[0]); ax[0].set_title('Age distribution'); ax[0].set_xlabel('age')
band = df[(df['age'] >= 13) & (df['age'] <= 21)]
band['age'].round().value_counts().sort_index().plot.bar(ax=ax[1])
ax[1].set_title(f'Boundary 13–21 coverage (n={len(band)})')
plt.tight_layout(); plt.savefig(FIG / 'age_distribution.png', dpi=120); plt.show()

In [ ]:
# Skin band + dataset + gender balance
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df['skin_band'].value_counts().plot.bar(ax=ax[0], title='Skin band')
df['dataset'].value_counts().plot.bar(ax=ax[1], title='Dataset')
df['gender'].value_counts().plot.bar(ax=ax[2], title='Gender')
plt.tight_layout(); plt.savefig(FIG / 'subgroup_balance.png', dpi=120); plt.show()

In [ ]:
# Fairness: dark-skin coverage across the boundary band + report
pivot = band.pivot_table(index='skin_band', columns='is_minor', values='image_id', aggfunc='count', fill_value=0)
dark = df[df['skin_band'].isin(['brown', 'dark'])]
print('boundary counts by skin_band x is_minor:\n', pivot)
print(f"dark-skin rows: {len(dark)} ({100*len(dark)/max(len(df),1):.1f}%)")

lines = ['# Kámárí Data Quality Report (v0)\n',
  f'- Total rows: {len(df)}', f'- Exact-age rows: {int((df.age_exact==True).sum())}',
  f'- Minors (age<18): {int((df.is_minor==True).sum())}', f'- Boundary 13–21 rows: {len(band)}',
  f'- Dark/brown skin rows: {len(dark)} ({100*len(dark)/max(len(df),1):.1f}%)',
  '\n## Datasets\n', df['dataset'].value_counts().to_markdown(),
  '\n## Skin band\n', df['skin_band'].value_counts().to_markdown(),
  '\n## Actions', '- [ ] Confirm dark-skin coverage for boundary fairness eval.',
  '- [ ] Oversample 13–21 during CNN training.']
(REPO / 'data' / 'cards' / 'data_quality_report.md').write_text('\n'.join(map(str, lines)))
print('\nwrote data/cards/data_quality_report.md\nNext: 04_push_to_huggingface.ipynb')